In [1]:
import os
import torch
import torch.nn as nn
from torchvision.transforms import v2 as transforms
import librosa
import numpy as np
from sklearn.metrics import roc_auc_score, mean_squared_error

In [2]:
generator = torch.Generator().manual_seed(42)
np.random.seed(42)

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [4]:
transform = transforms.RandomCrop(size=(128, 256))


class AudioDataset(torch.utils.data.Dataset):
    def __init__(self, audio_dir, train):
        self.audio_dir = audio_dir
        file_list = os.listdir(audio_dir)

        labels = np.zeros(len(file_list), dtype=int) if train else [1 if el[0] == 'a' else 0 for el in file_list]
        self.labels = torch.tensor(labels, dtype=torch.int8).to(device)

        loads = [librosa.load(os.path.join(audio_dir, el), sr=None) for el in file_list]
        spectrograms = [librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=128) for audio, sr in loads]
        spect_dbs = [
            torch.tensor(
                librosa.power_to_db(spec, ref=np.max),
                dtype=torch.float32
            )
            for spec in spectrograms
        ]

        spect_dbs = torch.stack(spect_dbs)

        mean = spect_dbs.mean(dim=0)
        std = spect_dbs.std(dim=0)

        self.spect_dbs = transform((spect_dbs - mean) / std)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.spect_dbs[idx], self.labels[idx]

In [5]:
train_dataset = AudioDataset('archive/dev_data/dev_data/slider/train', train=True)
test_dataset = AudioDataset('archive/dev_data/dev_data/slider/test', train=False)

In [6]:
batch_size = 32

train_set, validation_set = torch.utils.data.random_split(train_dataset, [0.8, 0.2], generator=generator)

train_loader = torch.utils.data.DataLoader(
    train_set,
    batch_size=batch_size,
    shuffle=True
)
validation_loader = torch.utils.data.DataLoader(
    validation_set,
    batch_size=batch_size,
    shuffle=False
)
test_loader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

In [7]:
class CNNAE(nn.Module):
    def __init__(self):
        super(CNNAE, self).__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 32, 5, stride=1, padding=2),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Conv2d(32, 64, 3, stride=1, padding=2),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Conv2d(64, 128, 5, stride=1, padding=0),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Conv2d(128, 256, 7, stride=1, padding=0),
            nn.ReLU(),
            nn.Dropout(0.3),
        )
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(256, 128, 7, stride=1, padding=0),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.ConvTranspose2d(128, 64, 5, stride=1, padding=0),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.ConvTranspose2d(64, 32, 3, stride=1, padding=2),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.ConvTranspose2d(32, 1, 5, stride=1, padding=2),
        )

    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.encoder(x)
        x = self.decoder(x)
        return x.squeeze(1)

In [8]:
def train_one_epoch(model, data_loader, optimizer, criterion):
    model.train()

    total_loss = 0

    for inputs, _ in data_loader:
        inputs = inputs.to(device)

        optimizer.zero_grad()

        outputs = model(inputs)
        # print(outputs.shape)
        # print(inputs.shape)

        loss = criterion(outputs, inputs)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(data_loader)


def validate(model, data_loader, criterion):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for inputs, _ in data_loader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, inputs)
            total_loss += loss.item()

    return total_loss / len(data_loader)


def compute_reconstruction_errors(model, data_loader):
    model.eval()
    errors = []
    with torch.no_grad():
        for batch, _ in data_loader:
            batch = batch.to(device)
            reconstructed = model(batch)
            # print(reconstructed.shape, batch.shape)
            error = torch.mean(((reconstructed - batch) ** 2).view(batch.size(0), -1), dim=1)
            # print(error.shape)
            errors.extend(error.cpu().numpy())
    return errors


def test(model, data_loader):
    model.eval()

    errors = compute_reconstruction_errors(model, data_loader)

    roc_auc_scores = roc_auc_score(test_dataset.labels.cpu(), errors)
    print(f"ROC AUC Score test: {roc_auc_scores}")

    return roc_auc_scores

In [9]:
def train(lr, weight_decay, epochs):
    model = CNNAE().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.MSELoss()

    for epoch in range(epochs):
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion)
        val_loss = validate(model, validation_loader, criterion)
        test(model, test_loader)

        print(f"Epoch {epoch + 1}/{epochs}, Train Loss: {train_loss:.4f}, Validation Loss: {val_loss:.4f}")

    return model

In [10]:
# model = CNNAE(input_dim=1, hidden_dim=32, latent_dim=64).to(device)
# optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-8)
# criterion = nn.MSELoss()
# train_one_epoch(model, train_loader, optimizer, criterion)
# validate(model, validation_loader, criterion)

In [11]:
train_dataset.spect_dbs.shape, test_dataset.spect_dbs.shape

(torch.Size([2370, 128, 256]), torch.Size([1101, 128, 256]))

In [12]:
model = train(lr=1e-3, weight_decay=1e-8, epochs=100)

KeyboardInterrupt: 

In [ ]:
# path = 'archive/dev_data/dev_data/slider/train'

# loads = [librosa.load(os.path.join(path, el), sr=None) for el in os.listdir(path)]
# print("Loaded")
# spectrograms = [librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=128) for audio, sr in loads]
# print("Spectrograms")
# not_flat = [
#     torch.tensor(
#         librosa.power_to_db(spec, ref=np.max),
#         dtype=torch.float32
#     )
#     for spec in spectrograms
# ]
# spect_dbs_not_flat = torch.stack(not_flat).to(device)

# flat = [
#     torch.tensor(
#         librosa.power_to_db(spec, ref=np.max),
#         dtype=torch.float32
#     ).flatten()
#     for spec in spectrograms
# ]
# spect_dbs_flat = torch.stack(flat).to(device)

In [ ]:
# mean_flat = torch.mean(spect_dbs_flat, dim=0)
# std_flat = torch.std(spect_dbs_flat, dim=0)

# end_flat = (spect_dbs_flat - mean_flat) / std_flat

In [ ]:
# mean_not_flat = torch.mean(spect_dbs_not_flat, dim=0)
# std_not_flat = torch.std(spect_dbs_not_flat, dim=0)
# end_not_flat = (spect_dbs_not_flat - mean_not_flat) / std_not_flat
# end2 = end_not_flat.flatten(start_dim=1)

In [ ]:
# torch.allclose(end_flat, end2)

In [ ]:
# spect_dbs_not_flat.shape